# Notebook 1 (Fase 1) — Ingesta de Datos → MongoDB
### Ernesto Investing AI · iDeSo · UNMSM — Semana 13

**Objetivo:**

Descargar datos reales de los 5 tickers mineros con `yfinance`, calcular indicadores técnicos (SMA, EMA, RSI), y almacenarlos en MongoDB Atlas.

**Prerequisitos:** Token de conexión `MONGO_URI` guardado en Secrets de Colab (ver Ejercicio 3 — MongoDB CRUD).

**Salida:** Colección `precios_ohlcv` poblada con datos reales de los 5 tickers.

**Pipeline:** yfinance → corrección de MultiIndex → indicadores técnicos → MongoDB Atlas

In [ ]:
# Paso 1 — Instalar librerias necesarias
!pip install "pymongo[srv]" yfinance --quiet

print("Librerias instaladas correctamente")

In [ ]:
# Paso 2 — Conectar a MongoDB Atlas
# La contrasena NUNCA se escribe en texto plano en el notebook: se pide de forma
# oculta con getpass y se inserta en la URI de conexion en tiempo de ejecucion.
# Alternativa con Secrets de Colab:
# from google.colab import userdata; MONGO_URI = userdata.get('MONGO_URI')
from getpass import getpass
from pymongo import MongoClient
from pymongo.server_api import ServerApi

DB_USER = "gdev03_db_user"
DB_PASSWORD = getpass("Password de MongoDB Atlas: ")

MONGO_URI = (
    f"mongodb+srv://{DB_USER}:{DB_PASSWORD}"
    "@cluster0.sxjck8h.mongodb.net/?appName=Cluster0"
)

client = MongoClient(MONGO_URI, server_api=ServerApi('1'))

# Verificar la conexion con un ping, manejando errores de forma explicita
try:
    client.admin.command("ping")
    print("Conectado a MongoDB Atlas correctamente")
except Exception as e:
    raise Exception("Ocurrio el siguiente error al conectar: ", e)

db = client["ernesto_investing_ai"]

In [ ]:
# Paso 3 — Definir los 5 tickers mineros del proyecto
TICKERS = {
    "FSM": "Fortuna Silver Mines Inc.",
    "VOLCABC1.LM": "Volcan Compania Minera S.A.A.",
    "ABX.TO": "Barrick Gold Corporation",
    "BVN": "Compania de Minas Buenaventura",
    "BHP": "BHP Billiton Limited"
}

print("Tickers del proyecto:")
for t, nombre in TICKERS.items():
    print(f"  {t}: {nombre}")

In [ ]:
# Paso 4 — Funciones de indicadores tecnicos
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime

def calcular_sma(serie, periodo):
    """Media movil simple."""
    return serie.rolling(window=periodo).mean()

def calcular_ema(serie, periodo):
    """Media movil exponencial."""
    return serie.ewm(span=periodo, adjust=False).mean()

def calcular_rsi(serie, periodo=14):
    """Indice de fuerza relativa (RSI)."""
    delta = serie.diff()
    ganancia = (delta.where(delta > 0, 0)).rolling(window=periodo).mean()
    perdida = (-delta.where(delta < 0, 0)).rolling(window=periodo).mean()
    rs = ganancia / perdida
    return 100 - (100 / (1 + rs))

print("Funciones de indicadores definidas: SMA, EMA, RSI")

In [ ]:
# Paso 5 — Funcion de ingesta para un ticker
def ingestar_ticker(ticker, periodo="1y"):
    """Descarga OHLCV de yfinance, calcula indicadores y guarda en MongoDB."""
    df = yf.download(ticker, period=periodo, auto_adjust=True, progress=False)

    if df.empty:
        print(f"  [{ticker}] Sin datos, se omite.")
        return 0

    # IMPORTANTE: yfinance devuelve un MultiIndex en las columnas incluso
    # al descargar un solo ticker. Hay que aplanarlo quedandonos con el
    # primer nivel (Open, High, Low, Close, Volume).
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Indicadores tecnicos
    df["sma_20"] = calcular_sma(df["Close"], 20)
    df["sma_50"] = calcular_sma(df["Close"], 50)
    df["ema_12"] = calcular_ema(df["Close"], 12)
    df["ema_26"] = calcular_ema(df["Close"], 26)
    df["rsi_14"] = calcular_rsi(df["Close"], 14)

    # Convertir cada fila a un documento de MongoDB
    def limpiar(v):
        # Reemplaza NaN por None para que el documento sea JSON valido
        return None if pd.isna(v) else round(float(v), 4)

    registros = []
    for fecha, fila in df.iterrows():
        registros.append({
            "ticker": ticker,
            "fecha": fecha.strftime("%Y-%m-%d"),
            "open": limpiar(fila["Open"]),
            "high": limpiar(fila["High"]),
            "low": limpiar(fila["Low"]),
            "close": limpiar(fila["Close"]),
            "volume": int(fila["Volume"]) if not pd.isna(fila["Volume"]) else 0,
            "sma_20": limpiar(fila["sma_20"]),
            "sma_50": limpiar(fila["sma_50"]),
            "ema_12": limpiar(fila["ema_12"]),
            "ema_26": limpiar(fila["ema_26"]),
            "rsi_14": limpiar(fila["rsi_14"]),
            "created_at": datetime.now()
        })

    # Se borran registros previos del ticker para evitar duplicados al re-ejecutar
    db["precios_ohlcv"].delete_many({"ticker": ticker})
    db["precios_ohlcv"].insert_many(registros)

    print(f"  [{ticker}] {len(registros)} dias guardados en MongoDB")
    return len(registros)

print("Funcion ingestar_ticker definida")

In [ ]:
# Paso 6 — Ingestar los 5 tickers
print("Iniciando ingesta de los 5 tickers...")
print()

total = 0
for ticker in TICKERS.keys():
    total += ingestar_ticker(ticker, periodo="1y")

print()
print(f"Ingesta completa: {total} documentos en total")
print("Tickers en la base:", db["precios_ohlcv"].distinct("ticker"))

In [ ]:
# Paso 7 — Celda de verificacion final
print("Resumen de la coleccion precios_ohlcv:")
print()

resumen_ok = True
for ticker in TICKERS.keys():
    n = db["precios_ohlcv"].count_documents({"ticker": ticker})
    ultimo = db["precios_ohlcv"].find_one({"ticker": ticker}, sort=[("fecha", -1)])
    if ultimo and ultimo.get("close") is not None:
        cierre = ultimo["close"]
        fecha = ultimo["fecha"]
        print(f"  [OK] {ticker}: {n} dias | ultimo cierre: ${cierre:.2f} ({fecha})")
    else:
        print(f"  [FALTA] {ticker}: no se encontraron datos")
        resumen_ok = False

print()
if resumen_ok:
    print("Verificacion exitosa: los 5 tickers tienen datos en MongoDB.")
else:
    print("Atencion: revisar los tickers marcados como [FALTA] antes de continuar al Notebook 2.")

## Resultado

La colección `precios_ohlcv` contiene datos reales de los 5 tickers mineros con sus indicadores técnicos (SMA-20, SMA-50, EMA-12, EMA-26, RSI-14).

El **Notebook 2 (Clasificador SVC)** leerá estos datos para entrenar el modelo, y la **API FastAPI (Fase 2)** los servirá al frontend.

**Pipeline:** yfinance → indicadores → **MongoDB** ✓